In [1]:
import random
import pandas as pd
import numpy as np
from astropy import units as u
from astropy.time import Time
from astropy.coordinates import solar_system_ephemeris, get_body_barycentric
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import math
from poliastro.bodies import Sun, Earth
from poliastro.twobody import Orbit
from poliastro.plotting import OrbitPlotter
from mpl_toolkits.mplot3d import Axes3D


import json
import random
from pathlib import Path
import pandas as pd


In [2]:
CWD = Path().cwd().parent
DATASET_DIR = CWD / 'Datasets'
nasa_csv = DATASET_DIR / 'Processed Datasets' / 'NASA+EuropeanSpaceAgency_NEO_Data_Cleaned.csv'

try:
    df = pd.read_csv(nasa_csv)
except FileNotFoundError:
    print(f"Error: The file {nasa_csv} was not found.")
    exit(1)
    
df

,Absolute_Magnitude,Est_Dia_In_Km_Min,Est_Dia_In_Km_Max,Close_Approach_Date,Relative_Velocity_Km_Per_Hr,Miss_Dist_Kilometers,Minimum_Orbit_Intersection,Jupiter_Tisserand_Invariant,Epoch_Osculation,Eccentricity,...,Inclination,Asc_Node_Longitude,Orbital_Period,Perihelion_Distance,Perihelion_Arg,Aphelion_Dist,Perihelion_Time,Mean_Anomaly,Mean_Motion,Hazardous
0,21.6,0.127220,0.284472,1995-01-01,22017.003799,6.275369e+07,0.025282,4.634000,2458000.5,0.425549,...,6.025981,314.373913,609.599786,0.808259,57.257470,2.005764,2.458162e+06,264.837533,0.590551,1
1,21.3,0.146068,0.326618,1995-01-01,65210.346095,5.729815e+07,0.186935,5.457000,2458000.5,0.351674,...,28.412996,136.717242,425.869294,0.718200,313.091975,1.497352,2.457795e+06,173.741112,0.845330,0
2,20.3,0.231502,0.517654,1995-01-08,27326.560182,7.622912e+06,0.043058,4.557000,2458000.5,0.348248,...,4.237961,259.475979,643.580228,0.950791,248.415038,1.966857,2.458120e+06,292.893654,0.559371,1
3,27.4,0.008801,0.019681,1995-01-15,40225.948191,4.268362e+07,0.005512,5.093000,2458000.5,0.216578,...,7.905894,57.173266,514.082140,0.983902,18.707701,1.527904,2.457902e+06,68.741007,0.700277,0
4,21.6,0.127220,0.284472,1995-01-15,35426.991794,6.101082e+07,0.034798,5.154000,2458000.5,0.210448,...,16.793382,84.629307,495.597821,0.967687,158.263596,1.483543,2.457814e+06,135.142133,0.726395,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13082,19.2,0.280000,0.280000,2038-02-11,73469.438027,7.290000e-03,0.001240,3.420095,2461000.5,0.708000,...,5.900000,145.252900,1072.300000,0.599000,269.934500,3.502000,2.461128e+06,317.211390,0.335727,0
13083,20.4,0.220000,0.500000,2040-08-12,71055.708102,8.973000e-02,0.087380,4.567631,2461000.5,0.670000,...,13.900000,119.067600,578.200000,0.448000,314.524000,2.269000,2.461228e+06,218.476860,0.622622,0
13084,20.6,0.200000,0.400000,2031-02-05,77305.646591,1.997000e-01,0.165970,5.569236,2461000.5,0.343000,...,32.900000,142.759400,405.600000,0.704000,49.386900,1.441000,2.460908e+06,81.896750,0.887574,0
13085,20.1,0.250000,0.600000,2065-08-17,76708.481085,1.226200e-01,0.106790,2.985408,2461000.5,0.863000,...,12.200000,296.783100,1211.600000,0.305000,150.776500,4.144000,2.461044e+06,347.798130,0.297128,0


In [3]:
# randomly pick a number between 0 and 684
random_index = random.randint(0, len(df) - 1)
asteroid = df.iloc[random_index]

# Extract orbital elements
a = asteroid['Semi_Major_Axis'] * u.AU  # Semi-major axis
e = asteroid['Eccentricity'] * u.one
i = asteroid['Inclination'] * u.deg
raan = asteroid['Asc_Node_Longitude'] * u.deg
argp = asteroid['Perihelion_Arg'] * u.deg
nu = asteroid['Mean_Anomaly'] * u.deg

# Number of points to simulate the orbit
num_points = 500
theta = np.linspace(0, 2*np.pi, num_points)

# Orbit in the orbital plane
r = a.to(u.km).value * (1 - e.value**2) / (1 + e.value * np.cos(theta))
x_orb = r * np.cos(theta)
y_orb = r * np.sin(theta)
z_orb = np.zeros_like(x_orb)

# Rotation matrices to convert to ecliptic coordinates
def rot_matrix(axis, angle_deg):
    angle = np.radians(angle_deg)
    if axis == 'x':
        return np.array([[1, 0, 0],
                         [0, np.cos(angle), -np.sin(angle)],
                         [0, np.sin(angle),  np.cos(angle)]])
    elif axis == 'z':
        return np.array([[np.cos(angle), -np.sin(angle), 0],
                         [np.sin(angle),  np.cos(angle), 0],
                         [0, 0, 1]])

# Apply rotations: Arg of Perihelion -> Inclination -> RAAN
coords = np.vstack((x_orb, y_orb, z_orb))
coords = rot_matrix('z', argp.value) @ coords
coords = rot_matrix('x', i.value) @ coords
coords = rot_matrix('z', raan.value) @ coords
x, y, z = coords

# Create 3D plot with Plotly
fig = go.Figure()

# Sun at origin
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0],
                           mode='markers',
                           marker=dict(size=10, color='yellow'),
                           name='Sun'))

# Asteroid orbit
fig.add_trace(go.Scatter3d(x=x, y=y, z=z,
                           mode='lines',
                           line=dict(color='red' if asteroid["Hazardous"] == 1 else 'green'),
                           name='Asteroid Orbit'))

# Earth orbit (approx circular, 1 AU)
theta_e = np.linspace(0, 2*np.pi, num_points)
x_e = np.cos(theta_e) * 1.0 * 1.496e8  # km
y_e = np.sin(theta_e) * 1.0 * 1.496e8
z_e = np.zeros_like(x_e)

fig.add_trace(go.Scatter3d(x=x_e, y=y_e, z=z_e,
                           mode='lines',
                           line=dict(color='blue'),
                           name='Earth Orbit'))

isHazardous = 'True' if asteroid["Hazardous"] == 1 else 'False'

fig.update_layout(scene=dict(
    xaxis_title='X (km)',
    yaxis_title='Y (km)',
    zaxis_title='Z (km)'),
    title=f'Asteroid Orbit Visualization, Hazard: {isHazardous}, click and drag to move around space'
)

fig.show()
